# Categorical rolling average: experiment (first used for employment status, ticket 2260)

Scores each split and window of the rolling average on known values, then re-scores the best on a temporal hide. Results go to S3 under `RESULTS_PREFIX/RUN_LABEL/`, one file per variant, so an interrupted run resumes where it stopped.

- **Test A (workplace folds):** held-out workplaces are removed from the rolling average inputs, then scored on their own known shares. Every variant runs.
- **Test B (temporal tail hide):** each workplace's last 12 months with known shares are hidden, the pipeline's imputation runs with the variant's rolling average, and the full imputed shares are scored on the hidden values. Runs for the top variants from test A plus the current setup.

**Before running:** set `DATA_BUCKET` and `RESULTS_PREFIX`. **Stop the notebook instance when finished.** Writes only to `RESULTS_PREFIX`.

Shares are scored with every row counting equally, on rows with all 5 known shares. Reusable for other categoricals (such as gender): see `README.md` in this folder; only the settings block in `experiment_utils.py` and the parameters below change.

## 0. Setup (SageMaker only)

First run `sagemaker_setup.ipynb` (with `BRANCH` set to the branch holding this code, `main` once merged) to clone or update the repo into `~/SageMaker/DataEngineering`. Do that whenever you want the latest code. This notebook only reads that clone, so Save As this notebook into your own folder first (the clone is reset on every setup run).

Run the next cell once, then restart the kernel.

In [ ]:
%pip install polars==1.44.1 pyarrow boto3

## 1. Parameters

- `DATA_BUCKET`: the bucket holding the cleaned employment status data (read only), `sfc-<branch name>-datasets`.
- `RESULTS_PREFIX`: where results are written; `RUN_LABEL` names this run's folder.
- `VARIANT_NAMES`: a list of variant names to run (such as `["current_6mo"]`), or `None` for all.
- `TEST_B_TOP_N`: how many of the best test A variants (lowest MAE) also run test B.
- `SAMPLE_PERCENT`: the share of workplaces (a fixed random sample) used for jumpiness and test B imputation.
- `LOCAL_ROOT`: set only to dry-run against local files; `None` on SageMaker.
- `REPO_DIR`: the clone made by `sagemaker_setup.ipynb`.

In [ ]:
DATA_BUCKET = "sfc-2260-empstat-avg-datasets"
CLEANED_DATASET = "domain=03_ind_cqc/dataset=02_employment_status_02_clean/"
RESULTS_PREFIX = "s3://skillsforcare-nonprod/sagemaker-notebooks/empstat-rolling-avg/"
RUN_LABEL = "run1"
VARIANT_NAMES = None
TEST_B_TOP_N = 3
SAMPLE_PERCENT = 10
N_FOLDS = 5
FOLD_SEED = 42
LOCAL_ROOT = None
REPO_DIR = "~/SageMaker/DataEngineering"

## Setup

Imports, the working copy of the prepared data, and the result locations.

In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path(REPO_DIR).expanduser().resolve()))

import experiments.categorical_rolling_average.experiment_utils as x
from polars_utils import utils as polars_utils
from projects._03_independent_cqc.utils import model_evaluation_utils as evaluation
from utils.column_names.ind_cqc_pipeline_columns import ModelEvaluationColumns as Eval

LOCAL = LOCAL_ROOT is not None
RESULTS_DIR = (
    str(Path(LOCAL_ROOT) / "results" / RUN_LABEL) if LOCAL else f"{RESULTS_PREFIX.rstrip('/')}/{RUN_LABEL}"
)
SOURCE = str(Path(LOCAL_ROOT) / "cleaned") if LOCAL else f"s3://{DATA_BUCKET}/{CLEANED_DATASET}"
PREPARED_PATH = Path("prepared_shares.parquet")
SELECTED_VARIANTS = [v for v in x.VARIANTS if VARIANT_NAMES is None or v.name in VARIANT_NAMES]
HIDDEN_PATH = Path("hidden_shares.parquet")

## 2. Prepare the data

Keeps only the columns needed, adds the workplace size band, the short-term imputed (trendline) shares and the workplace folds, then writes the result locally and scans it back. The size band uses the sum of `estimate_filled_posts_by_job_role` per workplace and date, which is present in the cleaned data. Rows are one per workplace, job role and date, so this table is large and is never held in memory.

In [ ]:
source_lf = pl.scan_parquet(f"{SOURCE}/*.parquet") if LOCAL else polars_utils.scan_parquet(SOURCE)
cleaned_lf = source_lf.select(
    x.LOCATION, x.DATE, x.SERVICE, x.REGION, x.ROLE, x.SIZE_POSTS, *x.KNOWN
).with_columns(pl.col(x.SERVICE, x.REGION, x.ROLE).cast(pl.Categorical))
cleaned_lf = x.add_size_band(cleaned_lf).drop(x.SIZE_POSTS)
cleaned_lf = x.add_trendline(cleaned_lf)
cleaned_lf = evaluation.assign_location_folds(cleaned_lf, x.LOCATION, N_FOLDS, FOLD_SEED)
cleaned_lf.sink_parquet(PREPARED_PATH)

prepared_lf = pl.scan_parquet(PREPARED_PATH)
print(prepared_lf.select(pl.len(), pl.col(x.LOCATION).n_unique()).collect())
print(prepared_lf.group_by(x.SIZE_BAND).len().sort(x.SIZE_BAND).collect())

## 2b. Summaries for scoring

Rolling averages only depend on the cell (service, region, size band, job role, date, fold), so they are built from per-cell totals. Three small tables are kept in memory: the cell totals, the rows with every share known (narrow, for scoring), and the rows of a fixed random sample of workplaces (for jumpiness).

In [ ]:
cells_df = x.build_cells(prepared_lf)
known_df = x.known_rows(prepared_lf)
sample_df = x.workplace_sample(
    prepared_lf, SAMPLE_PERCENT, [x.LOCATION, *x.CELL_DATE_KEYS, x.FOLD]
)
print(cells_df.height, known_df.height, sample_df.height)

## 3. Test A: held-out workplaces

One file per variant in `test_a/`, skipping any already written. Check `coverage` is 1.0 for every variant: the fallback to a coarser group gives every row a value.

In [ ]:
def variant_columns(df: pl.DataFrame, variant: x.Variant) -> pl.DataFrame:
    return df.with_columns(
        pl.lit(variant.name).alias("variant"),
        pl.lit(variant.split).alias("split"),
        pl.lit(variant.window_months).alias("window_months"),
    )


finished = set(x.list_results("test_a/", RESULTS_DIR, LOCAL))
for variant in SELECTED_VARIANTS:
    name = f"test_a/{variant.name}.parquet"
    if name in finished:
        continue
    predictions_df = x.predict_out_of_fold(cells_df, variant, N_FOLDS)
    scores_df = x.score_out_of_fold(predictions_df, known_df, sample_df)
    x.write_result(variant_columns(scores_df, variant), name, RESULTS_DIR, LOCAL)
    print(variant.name, "done")

## 4. Test A summary

Pooled scores per variant. `mean_absolute_error_pp` is lower-is-better; the skill scores are 1 minus the error over a pooled-mean baseline's error, so higher is better. Jumpiness is the mean absolute change between consecutive dates, in share units.

In [ ]:
def load_all(prefix: str) -> pl.DataFrame:
    return pl.concat(
        [x.read_result(name, RESULTS_DIR, LOCAL) for name in x.list_results(prefix, RESULTS_DIR, LOCAL)]
    )


def headline(results_df: pl.DataFrame) -> pl.DataFrame:
    return (
        results_df.filter(
            (pl.col(Eval.level).is_in(["pooled", "aggregate", "coverage", "jumpiness_workplace", "jumpiness_aggregate"]))
            & (pl.col(Eval.metric) != Eval.number_of_rows)
        )
        .with_columns((pl.col(Eval.level) + "_" + pl.col(Eval.metric)).alias("measure"))
        .pivot(on="measure", index="variant", values=Eval.value)
    )


test_a_df = load_all("test_a/")
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(60)
headline(test_a_df).sort(f"pooled_{Eval.mean_absolute_error_pp}")

## 5. Test B: temporal tail hide

Runs the top `TEST_B_TOP_N` variants by test A error, plus the current setup. Each workplace's last 12 months with known shares are hidden, and the pipeline's trendline imputation is re-run once on what is left (written to a local file). Each variant's rolling averages are then built from every workplace's hidden-data totals and applied to the sampled workplaces, where the pipeline's imputation runs and the imputed shares are scored on the hidden values. A variant that loses to the current setup here cannot win.

In [ ]:
ranked = (
    test_a_df.filter(
        (pl.col(Eval.level) == "pooled") & (pl.col(Eval.metric) == Eval.mean_absolute_error_pp)
    )
    .sort(Eval.value)
    .get_column("variant")
    .to_list()
)
variants_by_name = {v.name: v for v in x.VARIANTS}
test_b_variants = [variants_by_name[n] for n in ranked if n != x.CURRENT.name][:TEST_B_TOP_N] + [x.CURRENT]
print([v.name for v in test_b_variants])

hidden_lf = x.prepare_hidden_trendline(prepared_lf, HIDDEN_PATH)
hidden_cells_df = x.build_cells(hidden_lf)
hidden_sample_df = x.workplace_sample(
    hidden_lf,
    SAMPLE_PERCENT,
    [x.LOCATION, *x.CELL_DATE_KEYS, x.HIDDEN, *x.KNOWN, *x.TRUTH],
)
print(hidden_cells_df.height, hidden_sample_df.height)

Scores each test B variant: the rolling averages for every cell, then imputation and scoring on the sample. One file per variant in `test_b/`.

In [ ]:
finished = set(x.list_results("test_b/", RESULTS_DIR, LOCAL))
for variant in test_b_variants:
    name = f"test_b/{variant.name}.parquet"
    if name in finished:
        continue
    predictions_df = x.cell_predictions(hidden_cells_df.lazy(), variant, None).collect(engine="streaming")
    scores_df = x.score_hidden(predictions_df, hidden_sample_df)
    x.write_result(variant_columns(scores_df, variant), name, RESULTS_DIR, LOCAL)
    print(variant.name, "done")

## 6. Test B summary

Same measures as test A, on the hidden rows of the sampled workplaces. Coverage is the rolling average's, and jumpiness is of the imputed shares per sampled workplace. Open `compare.ipynb` for the win rule.

In [ ]:
headline(load_all("test_b/")).sort(f"pooled_{Eval.mean_absolute_error_pp}")